In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from ortools.linear_solver import pywraplp
import numpy as np
import pandas as pd
import sys
from typing import List, Dict, Union, Optional, Tuple
import random
import time
import json

import sys
sys.path.append("../")
from optimization_research.ml_constraints import MulticlassSolver, ItayChenSolverPreprocessIncluded, ItayChenSolver2, GonenSolver

In [3]:
def write_data(res, input_dataframe, prizes, limits,filename_towrite):
    with open(filename_towrite, 'a') as file:
        file.write('************************************************\n')
        json.dump(res.to_json(), file)
        file.write('\n')
        json.dump(limits, file)
        file.write('\n')
        json.dump(prizes, file)
        file.write('\n')
    input_dataframe.to_csv(filename_towrite, index=None, sep=' ', mode='a')

In [4]:
def cheak_results(all_results, prizes):
    df_prizes =  pd.DataFrame.from_dict(prizes,orient='index',columns =['prizes'])
    results = pd.merge(all_results,df_prizes, left_index=True, right_index=True).fillna(0)
    
    results['ItayChenSolverPreprocessIncluded_prizes'] = results['ItayChenSolverPreprocessIncluded'] * results['prizes']
    results['ItayChenSolver2_prizes'] = results['ItayChenSolver2'] * results['prizes']
    results['GonenSolver_prizes'] = results['GonenSolver'] * results['prizes']
    
    results = results.drop(columns=['ItayChenSolverPreprocessIncluded', 'ItayChenSolver2','GonenSolver'])
    return results.sum()

In [5]:
def solvers(limits, prizes, input_dataframe):
    solver1 = ItayChenSolverPreprocessIncluded(input_dataframe, limits = limits, prizes = prizes)
    solver2 = ItayChenSolver2(input_dataframe, limits = limits, prizes = prizes)
    solver3 = GonenSolver(input_dataframe, limits = limits, prizes = prizes)

    all_results = []
    all_times = {}
    for i, solver in enumerate([solver1, solver2, solver3]):
        if i == 0:
            name = 'ItayChenSolverPreprocessIncluded_time'
        elif i == 1:
            name = 'ItayChenSolver2_time'
        else:
            name = 'GonenSolver_time'
        start = time.time()
        r = solver.solve().rename(solver.__class__.__name__)
        end = time.time()
        all_times[name] = end - start
        print(f'time_calculted {solver} is : {end - start}')
        all_results.append(r)


    #concatenate the results
    all_results = pd.concat(all_results, axis = 1)
    all_times = pd.DataFrame.from_dict(all_times,  orient='index')
    return all_results,all_times.T

In [6]:
def create_test_data(n_users, n_credentials, n_providers, min_size , max_size , eq):
    #Divide the cardancial to users and providers
    input_dataframe = pd.DataFrame(columns = ["user", "credential", "provider", "credential_weight"])
    for cred in range(n_credentials):
        if cred < n_users:
            user = cred
        else:
            user = random.randint(1, n_users)
        if cred < n_providers:
            provider = cred
        else:
            provider = random.randint(1, n_providers)
        new_raw = {'user': f'user{user}', 'credential': f'credential{cred}', 'provider': f'provider{provider}', 'credential_weight': 1}
        input_dataframe = pd.concat([input_dataframe, pd.DataFrame([new_raw])], ignore_index = True)
    
    #genarte providers limits and users prizes
    limited_providers =  random.sample(range(n_providers), random.randint(n_providers * min_size, n_providers * max_size))
    limited_providers.sort()
    prized_users = random.sample(range(n_users), random.randint(n_users * min_size, n_users * max_size))
    prized_users.sort() 
    
    limits = {}
    if eq == 'Uniform from assigned':
        count = input_dataframe["provider"].value_counts().to_dict()
        for i in limited_providers:
            limits[f'provider{i}'] = int(count[f'provider{i}'] * random.random())
    elif eq == 'uniform from total':
        for i in limited_providers:
            limits[f'provider{i}'] = random.randint(0.05 * n_credentials, 0.15 * n_credentials)
    else:
        count = input_dataframe["provider"].value_counts().to_dict()
        for i in limited_providers:
            limits[f'provider{i}'] = np.random.chisquare(count[f'provider{i}']*0.8)

    prizes = {}
    for i in prized_users:
        prizes[f'user{i}'] = random.randint(1, 100)
    
    return input_dataframe, limits, prizes

In [7]:
#Set the diiffrent pramaters for the distributions
n_random_sizes = [(0.2,0.8),(0.8,1)]
random_eq_prvider_limit = ['Uniform from assigned', 'uniform from total', 'Chi squared']

## Run

In [ ]:
n_users = 100
n_credentials = 300
n_providers = 50
res_sum_df = pd.DataFrame(columns = ["prizes", "ItayChenSolverPreprocessIncluded_prizes", "ItayChenSolver2_prizes", "GonenSolver_prizes",
                                     "ItayChenSolverPreprocessIncluded_time","ItayChenSolver2_time","GonenSolver_time"])
for i in range(100):
    for eq in random_eq_prvider_limit:
        filename_towrite = f'results/comper_solvers/input_dataframe{eq}{sizes}{n_users}{n_credentials}{n_providers}.txt'
        for sizes in n_random_sizes:
            min_size = sizes[0]
            max_size = sizes[1]
            input_dataframe, limits, prizes = create_test_data(n_users, n_credentials, n_providers, min_size , max_size , eq)
            #solvers
            all_results, all_time = solvers(limits, prizes, input_dataframe)
            res = cheak_results(all_results, prizes)
            if (res[1] != res[2]) or (res[1] != res[3]) or (res[3] != res[2]):
                print('diffrent')
                write_data(res, input_dataframe, prizes, limits,filename_towrite)
            new_raw = {"prizes": res[0],"ItayChenSolverPreprocessIncluded_prizes": res[1],"ItayChenSolver2_prizes": res[2],"GonenSolver_prizes": res[3]}
            new = pd.concat([pd.DataFrame([new_raw]),all_time], axis = 1)
            res_sum_df = pd.concat([res_sum_df, new], ignore_index = True)
            res_sum_df['details'] = f"n_users = {n_users} , n_credentials = {n_credentials} ,n_providers = {n_providers} , sizes ditribtion = {sizes} , providers limits eq = {eq}"
            res_sum_df.to_csv(f"results/comper_solvers/reseach_solver{eq}{sizes}{n_users}{n_credentials}{n_providers}.csv")